# Exp1 作业：基于 Uni-Mol 的分子性质多目标回归

本 notebook 以 **Alchemy 分子性质数据集** 为例，演示如何使用 **Uni-Mol 预训练模型**，根据分子的原子类型和三维坐标同时预测多个分子性质标签，完成多目标回归任务。

**工作流概览：**

$$
\text{构建分子样本}
\rightarrow
\text{划分训练集与测试集}
\rightarrow
\text{微调 Uni-Mol 预训练模型}
\rightarrow
\text{预测测试集分子性质}
\rightarrow
\text{评估模型性能}
$$

**要求：** 按照各代码单元中的 `TODO` 提示补全代码，完成数据预处理、数据集划分、模型微调、测试集预测与性能评估等步骤。


## 1. 环境准备与数据集下载

### 1.1 环境依赖

本任务需要以下 Python 包：

| 包名 | 用途 |
|------|------|
| `pandas` | 读取、整理和筛选 CSV 表格数据 |
| `numpy` | 用于数组转换、数值计算，以及将多目标标签整理为矩阵形式 |
| `scikit-learn` | 用于数据集划分和模型评估，例如划分训练集/测试集、计算 MAE |
| `rdkit` | 用于读取 SDF 文件，解析分子的原子类型、键连接关系和三维构象坐标 |
| `unimol_tools` | 用于调用 Uni-Mol 相关接口，完成分子表示学习、模型微调和性质预测 |

如果上述包未全部安装，可使用以下命令统一安装：

```bash
pip install pandas numpy scikit-learn rdkit unimol_tools
```

> **(可运行下面代码进行自检)** 运行以下代码确认环境已配置完全：

In [ ]:
# 检查环境中是否已安装所需的 Python 包
import sys

packages_to_check = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "rdkit": "rdkit",
    "unimol_tools": "unimol_tools",
}

print("=" * 60)
print("检查环境中的 Python 包...")
print("=" * 60)

missing_packages = []

for display_name, import_name in packages_to_check.items():
    try:
        module = __import__(import_name)
        version = getattr(module, "__version__", "unknown")
        status = "✓ 已安装"
        print(f"{display_name:20s} | {status:12s} | 版本: {version}")
    except ImportError:
        status = "✗ 未安装"
        print(f"{display_name:20s} | {status:12s}")
        missing_packages.append(display_name)

print("=" * 60)

if missing_packages:
    print(f"\n⚠️  检测到 {len(missing_packages)} 个缺失的包:")
    for pkg in missing_packages:
        print(f"  - {pkg}")
    print("\n建议安装命令:")
    print(f"  pip install {' '.join(missing_packages)}")
else:
    print("\n✓ 所有必需的包都已安装，可以开始任务！")

### 1.2 数据集下载

作业使用 Alchemy Dataset，可从 `https://alchemy.tencent.com/data/alchemy-v20191129.zip` 下载后可解压至  `./data/Alchemy-v20191129/` 目录, 以方便后续代码加载使用。

或者，可以使用以下脚本命令自动完成下载与解压：

```bash
set -e
# 创建数据目录；如果目录已存在则不会报错
mkdir -p data
# 下载 Alchemy 数据集压缩包
wget -q --show-progress \
  -O data/alchemy-v20191129.zip \
  https://alchemy.tencent.com/data/alchemy-v20191129.zip
# 解压数据集到 data/ 目录
unzip -oq data/alchemy-v20191129.zip -d data/
# 删除压缩包，节省磁盘空间
rm data/alchemy-v20191129.zip
```

数据集的核心文件为：final_version.csv（分子标签和元数据）和 atom_N/*.sdf（对应分子的三维结构文件）。


> **(可运行下面代码进行自检)** 运行以下代码确认数据集已放置到正确路径，默认数据目录为`./data/Alchemy-v20191129/`。

In [ ]:
from pathlib import Path

DATA_DIR = Path("./data/Alchemy-v20191129")
CSV_PATH = DATA_DIR / "final_version.csv"

print("CSV exists:", CSV_PATH.exists())
print("Data dir exists:", DATA_DIR.exists())
print("Data dir:", DATA_DIR)
print("CSV path:", CSV_PATH)

assert DATA_DIR.exists(), f"找不到数据目录: {DATA_DIR}"
assert CSV_PATH.exists(), f"找不到标签文件: {CSV_PATH}"


### 1.3 采样配置

为方便调试和正式训练之间切换，这里提供一个简单的采样开关。通过修改 `SAMPLE_N`，可以控制后续实际使用的数据量。

| 参数值 | 含义 |
|--------|------|
| `SAMPLE_N = 1000` | 随机抽取 1000 个样本，用于快速测试流程 |
| `SAMPLE_N = None` | 使用完整数据集，不进行采样 |

后续读取数据时会根据该参数决定是否抽样。调试代码时建议先使用较小的样本数；确认流程无误后，再切换为全量数据训练。

In [ ]:
SAMPLE_N = 1000  # 改为 None 以使用全量数据集
RANDOM_STATE = 42

print(f"采样配置: SAMPLE_N={SAMPLE_N}, RANDOM_STATE={RANDOM_STATE}")


## 2. 数据处理
### 2.1 原始数据介绍
为了后续准确构建 Uni-Mol 所需的输入格式，我们首先需要理解原始数据的结构。下表展示了 CSV 文件的关键列及其含义：

| 列名 | 数据类型 | 描述 | 用途 |
|------|------|------|------|
| `gdb_idx` | int | 分子的全局唯一标识符 | 用于定位对应的 SDF 三维结构文件 |
| `atom_number` | int | 分子中的原子总数 | 用于定位 `atom_N/` 目录下的文件夹 |
| `zpve`、`Cv`、`gap` 等 | float | 12 个分子性质的标签值 | 用于构建多目标回归的 target 向量 |

原始标签文件中部分列名包含空格、下划线不一致或单位说明，例如 atom number、zpve\n(Ha, zero point vibrational energy) 和 Cv\n(cal/molK, heat capacity at 298.15 K)。为便于后续代码处理，建议对这些列名进行统一和简化：分子标识符统一为 gdb_idx，原子数统一为 atom_number，各性质标签则简化为更短的目标列名。处理后最终保留 12 个分子性质目标，分别为：zpve、Cv、gap、G、HOMO、U、alpha、U0、H、LUMO、mu 和 R2。

处理后表头示例：

| gdb_idx | atom_number | zpve | Cv | gap | G | HOMO | U | alpha | U0 | H | LUMO | mu | R2 |
|---------|-------------|------|-----|------|-------|--------|-------|---------|-------|-------|--------|-------|---------|


> **(需手动运行下一个单元格代码)** 


In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

rename_map = {
    "gdb_id": "gdb_idx",
    "atom number": "atom_number",
    "num_atoms": "atom_number",
    "zpve\n(Ha, zero point vibrational energy)": "zpve",
    "Cv\n(cal/molK, heat capacity at 298.15 K)": "Cv",
    "gap\n(Ha, LUMO-HOMO)": "gap",
    "G\n(Ha, Free energy at 298.15 K)": "G",
    "HOMO\n(Ha, energy of HOMO)": "HOMO",
    "U\n(Ha, internal energy at 298.15 K)": "U",
    "alpha\n(a_0^3, Isotropic polarizability)": "alpha",
    "U0\n(Ha, internal energy at 0 K)": "U0",
    "H\n(Ha, enthalpy at 298.15 K)": "H",
    "LUMO\n(Ha, energy of LUMO)": "LUMO",
    "mu\n(D, dipole moment)": "mu",
    "R2\n(a_0^2, electronic spatial extent)": "R2",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

target_cols = [
    "zpve", "Cv", "gap", "G", "HOMO", "U",
    "alpha", "U0", "H", "LUMO", "mu", "R2",
]

required_cols = ["gdb_idx", "atom_number", *target_cols]
missing_cols = [col for col in required_cols if col not in df.columns]
assert not missing_cols, f"标签表缺少必要列: {missing_cols}"

if SAMPLE_N is not None and len(df) > SAMPLE_N:
    df = df.sample(n=SAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)

print("rows for this notebook:", len(df))
print("target dim:", len(target_cols))
display(df.head())

### 2.2 Uni-Mol 输入数据格式说明

Uni-Mol 要求输入数据采用统一的字典格式。对于多目标回归任务，该字典需包含三个关键字段：atoms、coordinates 和 target，分别存储原子信息、三维坐标和预测标签。

| 字段 | 类型 | 说明 | 示例 |
|------|------|------|------|
| `atoms` | list of str | 所有分子的原子符号列表 | `[["C", "H", "H", "H", "H"], ...]` |
| `coordinates` | list of ndarray | 所有分子各原子的三维坐标 | `[[[0.0, 0.0, 0.0], [...]], ...]` |
| `target` | list of list | 所有分子的标签向量（12 维） | `[[0.123, 0.456, ...], ...]` |

**示例：**

```python
# Uni-Mol 要求的输入格式（批量）
unimol_data = {
    "atoms": [sample1_atoms, sample2_atoms, ...],           # N 个分子
    "coordinates": [sample1_coords, sample2_coords, ...],   # N 个分子
    "target": [sample1_targets, sample2_targets, ...],      # N 个分子
}

# 单个分子的样本格式
example_sample = {
    "atoms": ["C", "H", "H", "H", "H"],           # 该分子有 5 个原子
    "coordinates": [
        [0.0000, 0.0000, 0.0000],                 # C 原子坐标
        [0.6291, 0.6291, 0.6291],                 # H 原子 1 坐标
        [-0.6291, -0.6291, 0.6291],               # H 原子 2 坐标
        [-0.6291, 0.6291, -0.6291],               # H 原子 3 坐标
        [0.6291, -0.6291, -0.6291],               # H 原子 4 坐标
    ],
    "targets": [0.123, 0.456, ..., 0.789]         # 12 个预测目标的标签值
}
```

### 2.3 构建样本列表（TODO）

为了将原始数据集整合成 Uni-Mol 需要的格式，我们需要用前面处理好的 DataFrame `df` 中的 `gdb_idx` 和 `atom_number` 字段，在 `atom_N/` 目录下找到对应的 SDF 文件，提取每个分子的原子类型和三维坐标，再结合 `df` 中的 12 个预测目标（`target_cols`），组装成包含 `atoms`、`coordinates` 和 `targets` 字段的样本列表。

其中，`df` 是前面数据处理步骤中读取并清洗后的 DataFrame（已完成列名统一、采样等操作），`target_cols` 是 12 个分子性质标签列表 `["zpve", "Cv", "gap", "G", "HOMO", "U", "alpha", "U0", "H", "LUMO", "mu", "R2"]`。

> 本单元需要生成 `samples` 列表。每个元素是一个字典，必须包含 `atoms`、`coordinates`、`targets` 三个字段。后续划分训练集和测试集时会直接使用 `samples`。

In [ ]:
from rdkit import Chem

samples = []

for _, row in df.iterrows():
    gdb_idx = int(row["gdb_idx"])
    atom_number = int(row["atom_number"])
    sdf_path = DATA_DIR / f"atom_{atom_number}" / f"{gdb_idx}.sdf"

    if not sdf_path.exists():
        continue

    # TODO 1: 读取 SDF 文件，得到 mol
    mol = ...

    if mol is None or mol.GetNumConformers() == 0:
        continue

    # TODO 2: 提取 atoms / coordinates / targets
    atoms = ...
    coordinates = ...
    targets = ...

    samples.append({
        "atoms": atoms,
        "coordinates": coordinates,
        "targets": targets,
    })

print("usable samples:", len(samples))
print("target dim:", len(samples[0]["targets"]) if samples else 0)

### 2.4 划分训练集和测试集（TODO）

按 9:1 比例随机划分样本为训练集和测试集，并分别转换为 Uni-Mol 字典格式。

训练集供 MolTrain 进行 5-fold 交叉验证微调，测试集用于模型评估。

> 本单元需要生成 train_data 和 test_data 两个 Uni-Mol 输入字典。二者都应包含 atoms、coordinates、target 三个键，其中 target 是形状为 (N, 12) 的标签数组或列表。



In [ ]:
from sklearn.model_selection import train_test_split

# TODO 1: 按 9:1 划分 samples
train_samples, test_samples = ...


def to_unimol_dict(sample_list):
    return {
        "atoms": ...,
        "coordinates": ...,
        "target": ...,
    }


# TODO 2: 转成 Uni-Mol 批量输入格式
train_data = ...
test_data = ...


print(f"train: {len(train_data['atoms'])}, test: {len(test_data['atoms'])}")
print("train target shape:", np.asarray(train_data["target"]).shape)
print("test target shape:", np.asarray(test_data["target"]).shape)


### 2.5 保存评估所需的真实标签

后续训练和预测会调用 `unimol_tools`。为了避免第三方工具在内部预处理数据时改变输入字典结构，本节先将训练集和测试集的真实标签单独保存下来。

其中，训练集标签的标准差 `train_target_std` 将用于后续计算 NMAE：

$$
\mathrm{NMAE}_i = \frac{\mathrm{MAE}_i}{\sigma_i^{train}}
$$

这样可以让不同量纲、不同数值范围的分子性质具有可比性。


In [ ]:
import numpy as np

y_train_true = np.asarray(train_data["target"], dtype=float)
y_test_true = np.asarray(test_data["target"], dtype=float)


print("y_train_true shape:", y_train_true.shape)
print("y_test_true shape:", y_test_true.shape)



## 3. 模型训练与评估

### 3.1 微调 Uni-Mol 模型 (TODO)

在前一步中，我们已将全部样本按 **9:1** 划分为训练集和测试集。本节仅使用训练集对 Uni-Mol 预训练模型进行微调，并通过 `kfold=5` 在训练集内部执行 5-fold 交叉验证，用于模型训练、验证与集成预测。测试集不参与训练和调参，仅用于最终泛化性能评估。

微调过程使用 `unimol_tools` 提供的 `MolTrain` 接口。模型以分子的原子类型和三维坐标作为输入，学习同时预测多个分子性质标签，从而完成多目标回归任务。

`MolTrain` 接口的详细参数说明可参见：[unimol-tools 官方文档](https://unimol-tools.readthedocs.io/en/latest/)

In [ ]:
from unimol_tools import MolTrain

RUN_NAME = "unimol_teaching_demo_seed42"
SAVE_PATH = f"./exp/{RUN_NAME}"

trainer = MolTrain(
    task="multilabel_regression",
    data_type="molecule",
    epochs=5,
    batch_size=32,
    kfold=5,
    metrics="mae",
    save_path=SAVE_PATH,
    model_name="unimolv1",
    remove_hs=False,
)

# TODO: 在 train_data 上微调模型
trainer.fit(data=...)

### 3.2 在测试集上进行预测

完成 3.1 的 5-fold 微调训练后，接下来使用 `unimol_tools` 提供的 `MolPredict` 接口加载训练阶段保存的模型，并对测试集分子进行性质预测。

由于训练时设置了 `kfold=5`，因此会得到 5 个子模型。`MolPredict` 会自动加载这些模型，对测试集分别进行推理，并对多个模型的预测结果进行 ensemble 平均，输出最终预测值。测试集不参与模型训练或参数选择，仅用于评估模型的泛化性能。

`MolPredict` 接口的详细参数说明可参见：[unimol-tools 官方文档](https://unimol-tools.readthedocs.io/en/latest/)

In [ ]:
from unimol_tools import MolPredict

predictor = MolPredict(
    load_model=SAVE_PATH,

)

# TODO 1: 准备只包含 atoms / coordinates 的测试输入
test_input = {
    "atoms": ...,
    "coordinates": ...,
}

# TODO 2: 调用 predictor.predict 得到 y_pred
y_pred = predictor.predict(data=...)
y_pred = np.asarray(y_pred, dtype=float)

print("y_pred shape:", y_pred.shape)

### 3.3 评估模型性能

完成测试集预测后，接下来将预测结果与真实标签进行比较，评估模型在 12 个分子性质目标上的整体预测性能。本实验采用 **NMAE**（Normalized Mean Absolute Error，归一化平均绝对误差）作为主要评价指标。相比直接平均不同目标的 MAE，NMAE 会先用训练集中对应目标的标准差进行归一化，从而减弱不同性质在单位和数值尺度上的差异。

对于第 $i$ 个目标，其 MAE 定义为：

$$
\text{MAE}_i =
\frac{1}{n}
\sum_{j=1}^{n}
\left|
y_{\text{true}}^{(j,i)} - y_{\text{pred}}^{(j,i)}
\right|
$$

其中，$n$ 表示测试样本数量，$y_{\text{true}}^{(j,i)}$ 和 $y_{\text{pred}}^{(j,i)}$ 分别表示第 $j$ 个样本在第 $i$ 个目标上的真实值和预测值。

随后，使用训练集中第 $i$ 个目标的标准差 $\sigma_i$ 对 MAE 进行归一化：

$$
\text{NMAE}_i =
\frac{\text{MAE}_i}{\sigma_i}
$$

最后，将 12 个目标的 NMAE 取平均，得到最终评价指标：

$$
\text{Final NMAE}
=
\frac{1}{12}
\sum_{i=1}^{12}
\text{NMAE}_i
$$

$\text{Final NMAE}$ 越小，说明模型在不同性质目标上的整体预测误差越低，预测性能越好。

In [ ]:
from sklearn.metrics import mean_absolute_error

y_train = y_train_true# 这里多余赋值是为了提醒后续计算需要这些参数
y_true = y_test_true
y_pred = np.asarray(y_pred, dtype=float)

mae_list = []
nmae_list = []

for i, target_name in enumerate(target_cols):
    # TODO: 计算第 i 个目标的 MAE、训练集标准差和 NMAE
    mae_i = ...
    std_i = ...#需要计算训练集的标准差std
    nmae_i = ...

    mae_list.append(mae_i)
    nmae_list.append(nmae_i)
    print(f"{target_name:>5s} | MAE: {mae_i:.6f} | NMAE: {nmae_i:.6f}")

final_nmae = np.mean(...)
print("Final NMAE:", final_nmae)